<a href="https://colab.research.google.com/github/953625244013-cell/RFM-CLV-Personalized-Email/blob/main/Copy_of_RFM_CLV_Personalized_Email_Generator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd

# Upload your CSV file
from google.colab import files
uploaded = files.upload()

# Read the uploaded file
df = pd.read_csv("customer_transactions.csv")

# Display the data
df.head()

Saving customer_transactions.csv to customer_transactions.csv


,CustomerID,PurchaseDate,OrderID,Category,Product,Amount
0,C001,2026-09-14,O001,Electronics,Wireless Headphones,4500
1,C001,2026-08-20,O002,Electronics,Keyboard,2200
2,C001,2026-07-12,O003,Electronics,Mouse,1200
3,C001,2026-06-05,O004,Electronics,Monitor,12500
4,C002,2026-09-05,O005,Fashion,Shoes,3500


In [ ]:
# STEP 2: RFM ANALYSIS

# Make sure PurchaseDate is in date format
df["PurchaseDate"] = pd.to_datetime(df["PurchaseDate"])

# Reference date
reference_date = pd.Timestamp("2026-09-24")

# Calculate RFM values
rfm = df.groupby("CustomerID").agg(
    Recency=("PurchaseDate", lambda x: (reference_date - x.max()).days),
    Frequency=("OrderID", "nunique"),
    Monetary=("Amount", "sum")
).reset_index()

# Display RFM values
rfm

,CustomerID,Recency,Frequency,Monetary
0,C001,10,4,20400
1,C002,19,3,8900
2,C003,175,2,9300
3,C004,278,1,900
4,C005,6,4,4650
5,C006,257,1,5500
6,C007,12,2,9800
7,C008,197,2,2800
8,C009,5,1,2500
9,C010,313,1,1500


In [ ]:
# STEP 3: Calculate RFM Scores

# Rank customers into 1-5 groups
rfm["R_Score"] = pd.qcut(
    rfm["Recency"].rank(method="first"),
    5,
    labels=[5, 4, 3, 2, 1]
).astype(int)

rfm["F_Score"] = pd.qcut(
    rfm["Frequency"].rank(method="first"),
    5,
    labels=[1, 2, 3, 4, 5]
).astype(int)

rfm["M_Score"] = pd.qcut(
    rfm["Monetary"].rank(method="first"),
    5,
    labels=[1, 2, 3, 4, 5]
).astype(int)

# Combined RFM score
rfm["RFM_Score"] = (
    rfm["R_Score"].astype(str) +
    rfm["F_Score"].astype(str) +
    rfm["M_Score"].astype(str)
)

rfm

,CustomerID,Recency,Frequency,Monetary,R_Score,F_Score,M_Score,RFM_Score
0,C001,10,4,20400,4,5,5,455
1,C002,19,3,8900,3,4,4,344
2,C003,175,2,9300,3,3,4,334
3,C004,278,1,900,1,1,1,111
4,C005,6,4,4650,5,5,3,553
5,C006,257,1,5500,2,1,3,213
6,C007,12,2,9800,4,3,5,435
7,C008,197,2,2800,2,4,2,242
8,C009,5,1,2500,5,2,2,522
9,C010,313,1,1500,1,2,1,121


In [ ]:
# STEP 4: Customer Segmentation

def segment_customer(row):
    r = row["R_Score"]
    f = row["F_Score"]
    m = row["M_Score"]

    if r >= 4 and f >= 4 and m >= 4:
        return "Champions"
    elif f >= 4 and m >= 3:
        return "Loyal Customers"
    elif r >= 4 and f <= 3:
        return "Potential Loyalists"
    elif r <= 2 and f >= 3 and m >= 3:
        return "At Risk"
    elif r <= 2 and f <= 2:
        return "Lost Customers"
    else:
        return "Others"

rfm["Segment"] = rfm.apply(segment_customer, axis=1)

# Display customer segments
rfm

,CustomerID,Recency,Frequency,Monetary,R_Score,F_Score,M_Score,RFM_Score,Segment
0,C001,10,4,20400,4,5,5,455,Champions
1,C002,19,3,8900,3,4,4,344,Loyal Customers
2,C003,175,2,9300,3,3,4,334,Others
3,C004,278,1,900,1,1,1,111,Lost Customers
4,C005,6,4,4650,5,5,3,553,Loyal Customers
5,C006,257,1,5500,2,1,3,213,Lost Customers
6,C007,12,2,9800,4,3,5,435,Potential Loyalists
7,C008,197,2,2800,2,4,2,242,Others
8,C009,5,1,2500,5,2,2,522,Potential Loyalists
9,C010,313,1,1500,1,2,1,121,Lost Customers


In [ ]:
# STEP 5: Calculate Customer Lifetime Value (CLV)

# Average Order Value
rfm["AOV"] = rfm["Monetary"] / rfm["Frequency"]

# Estimated purchase frequency
rfm["Purchase_Frequency"] = rfm["Frequency"]

# Simple CLV estimate
rfm["CLV"] = rfm["AOV"] * rfm["Purchase_Frequency"]

# Display CLV
rfm[[
    "CustomerID",
    "Segment",
    "Frequency",
    "Monetary",
    "AOV",
    "CLV"
]]

In [ ]:
# Check the columns currently available
print(rfm.columns.tolist())

['CustomerID', 'Recency', 'Frequency', 'Monetary', 'R_Score', 'F_Score', 'M_Score', 'RFM_Score', 'Segment', 'Promotion']


In [ ]:
# Calculate CLV

rfm["AOV"] = rfm["Monetary"] / rfm["Frequency"]

rfm["CLV"] = rfm["AOV"] * rfm["Frequency"]

print(rfm[["CustomerID", "AOV", "CLV"]])

  CustomerID          AOV      CLV
0       C001  5100.000000  20400.0
1       C002  2966.666667   8900.0
2       C003  4650.000000   9300.0
3       C004   900.000000    900.0
4       C005  1162.500000   4650.0
5       C006  5500.000000   5500.0
6       C007  4900.000000   9800.0
7       C008  1400.000000   2800.0
8       C009  2500.000000   2500.0
9       C010  1500.000000   1500.0


In [ ]:
# STEP 6: Personalized Promotion Generator

def get_promotion(segment):
    promotions = {
        "Champions": "20% OFF on your next purchase",
        "Loyal Customers": "15% OFF on selected products",
        "Potential Loyalists": "10% OFF to welcome you back",
        "At Risk": "25% OFF - We miss you!",
        "Lost Customers": "30% OFF - Come back and shop with us",
        "Others": "10% OFF on your next order"
    }

    return promotions.get(segment, "10% OFF on your next order")


rfm["Promotion"] = rfm["Segment"].apply(get_promotion)

rfm[["CustomerID", "Segment", "CLV", "Promotion"]]

,CustomerID,Segment,CLV,Promotion
0,C001,Champions,20400.0,20% OFF on your next purchase
1,C002,Loyal Customers,8900.0,15% OFF on selected products
2,C003,Others,9300.0,10% OFF on your next order
3,C004,Lost Customers,900.0,30% OFF - Come back and shop with us
4,C005,Loyal Customers,4650.0,15% OFF on selected products
5,C006,Lost Customers,5500.0,30% OFF - Come back and shop with us
6,C007,Potential Loyalists,9800.0,10% OFF to welcome you back
7,C008,Others,2800.0,10% OFF on your next order
8,C009,Potential Loyalists,2500.0,10% OFF to welcome you back
9,C010,Lost Customers,1500.0,30% OFF - Come back and shop with us


In [ ]:
# STEP 7: Personalized Email Generator

def generate_email(row):
    return {
        "customer_id": row["CustomerID"],
        "segment": row["Segment"],
        "clv": round(row["CLV"], 2),
        "subject": f"Special Offer Just for You, {row['CustomerID']}!",
        "greeting": f"Hi {row['CustomerID']},",
        "body": (
            "Thank you for being a valued customer. "
            "We have a special offer selected especially for you."
        ),
        "offer": row["Promotion"],
        "call_to_action": "Shop Now",
        "closing": "Thank you for shopping with us!"
    }

# Generate emails
emails = rfm.apply(generate_email, axis=1)

# Display the first customer's email
emails.iloc[0]

{'customer_id': 'C001',
 'segment': 'Champions',
 'clv': 20400.0,
 'subject': 'Special Offer Just for You, C001!',
 'greeting': 'Hi C001,',
 'body': 'Thank you for being a valued customer. We have a special offer selected especially for you.',
 'offer': '20% OFF on your next purchase',
 'call_to_action': 'Shop Now',
 'closing': 'Thank you for shopping with us!'}

In [ ]:
# STEP 8: Convert Personalized Emails to JSON

import json

email_list = emails.tolist()

# Convert to JSON format
json_output = json.dumps(email_list, indent=4)

# Display the JSON
print(json_output)

[
    {
        "customer_id": "C001",
        "segment": "Champions",
        "clv": 20400.0,
        "subject": "Special Offer Just for You, C001!",
        "greeting": "Hi C001,",
        "body": "Thank you for being a valued customer. We have a special offer selected especially for you.",
        "offer": "20% OFF on your next purchase",
        "call_to_action": "Shop Now",
        "closing": "Thank you for shopping with us!"
    },
    {
        "customer_id": "C002",
        "segment": "Loyal Customers",
        "clv": 8900.0,
        "subject": "Special Offer Just for You, C002!",
        "greeting": "Hi C002,",
        "body": "Thank you for being a valued customer. We have a special offer selected especially for you.",
        "offer": "15% OFF on selected products",
        "call_to_action": "Shop Now",
        "closing": "Thank you for shopping with us!"
    },
    {
        "customer_id": "C003",
        "segment": "Others",
        "clv": 9300.0,
        "subject": "Speci